Libraries

In [ ]:
pip install ultralytics

In [ ]:
import os
import pandas as pd
import nibabel as nib
import numpy as np
import nibabel as nib
import matplotlib.pyplot as plt
from scipy.ndimage import gaussian_filter, uniform_filter, median_filter
from skimage.metrics import peak_signal_noise_ratio as psnr
from skimage.metrics import structural_similarity as ssim
from skimage.transform import resize
from skimage.feature import canny, graycomatrix, graycoprops
from skimage.filters import sobel
from skimage.morphology import erosion, dilation, opening, closing, disk
from skimage.measure import find_contours, label, regionprops
from scipy.spatial import ConvexHull
from scipy.ndimage import binary_fill_holes
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score
from sklearn.cluster import KMeans
from sklearn.preprocessing import StandardScaler
from ultralytics import YOLO
import cv2
import csv
from sklearn.neighbors import KNeighborsClassifier
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.model_selection import cross_validate, StratifiedKFold
from IPython.display import Image, display

Importing dataset

In [ ]:
dataset_path = '/content/drive/MyDrive/brats ped dataset 25 samples'
patients = os.listdir(dataset_path)
print(f"Patients found: {patients}")

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

# ***E-assignment 01***

### Step 1. batch processing

In [ ]:
images = {} #dictionary to store all loaded images
selected_slices = {} #dictionary to store selected slice index for each patient
dropped_slices = [] #list of dicts to log dropped slices

for patient in sorted(os.listdir(dataset_path)):
    patient_path = os.path.join(dataset_path, patient)
    if os.path.isdir(patient_path):
        images[patient] = {}

        # Load T2-FLAIR image to find the slice with the highest intensity variance of brain tissue
        t2f_file = [f for f in os.listdir(patient_path) if 't2f' in f and (f.endswith('.nii.gz') or f.endswith('.nii'))]
        if not t2f_file:
            print(f"Skipping {patient}: No T2-FLAIR file found.")
            continue

        t2f_data_path = os.path.join(patient_path, t2f_file[0])
        t2f_data = nib.load(t2f_data_path).get_fdata()

        # Calculate tissue intensity threshold (intensity > 10% of global max in volume)
        global_max = np.max(t2f_data)
        thresh = 0.1 * global_max

        # Prune slice search range to exclude lower face/neck and top-scalp artifacts
        num_slices = t2f_data.shape[2]
        start_slice = int(num_slices * 0.15) # Starts around slice 23 (above neck/mouth)
        end_slice = int(num_slices * 0.80)   # Ends around slice 124 (below scalp fat)

        # Count brain voxels for each slice
        counts = [np.sum(t2f_data[:, :, idx] > thresh) for idx in range(num_slices)]
        max_count = np.max(counts)

        # Candidate slices must contain at least 50% of the maximum brain cross-section area
        min_pixels = 0.5 * max_count

        best_slice_idx = -1
        max_variance = -1.0

        for i in range(start_slice, end_slice):
            slice_data = t2f_data[:, :, i]
            brain_pixels = slice_data[slice_data > thresh]
            if len(brain_pixels) >= min_pixels:
                var = np.var(brain_pixels)
                if var > max_variance:
                    max_variance = var
                    best_slice_idx = i

        if best_slice_idx == -1:
            best_slice_idx = num_slices // 2

        selected_slices[patient] = best_slice_idx
        print(f"Patient: {patient} | Unsupervised Selected T2-FLAIR Slice: {best_slice_idx} (Variance: {max_variance:.4f}, Brain Pixels: {len(t2f_data[:, :, best_slice_idx][t2f_data[:, :, best_slice_idx] > thresh])})")

        # Load the 2D slice for all modalities at best_slice_idx
        for file in sorted(os.listdir(patient_path)):
            if file.endswith('.nii.gz') or file.endswith('.nii'):
              if "seg" in file: #we will skip the segmentation mask files here
                continue
              modality = file.split('_')[-1].replace('.nii.gz', '').replace('.nii', '')
              filepath = os.path.join(patient_path, file)
              img = nib.load(filepath)
              images[patient][modality] = img.get_fdata()[:, :, best_slice_idx]
              print(f"Loaded: {patient} | {modality} | Shape: {images[patient][modality].shape}")

print(f"\nTotal patients loaded: {len(images)}")

### Step 2: Radiometric Resolution (bit-depth detection)

we'll find the bit depth for each of the 4 slices in each of the 25c patient folders

In [ ]:
modalities_to_process = ['t1n','t1c', 't2w', 't2f']

for patient, modalities in images.items(): #going thru every patient and their 4 MRI scans that we loaded in prev step
    print(f"\n{patient}:")
    for modality in modalities_to_process:
      # Construct the correct key for the image data
        image_key = f"{patient}-{modality}"
      #getting basic stats. min_val/max_val means the darkest and brightest pixel values in the scan
        data = modalities[image_key]
        min_val = data.min()
        max_val = data.max()
        unique_vals = len(np.unique(data)) #unique_val is how many distinct intensity values exist in the image

        #determine bit depth
        if max_val <= 255:
            bit_depth = "8-bit"
        elif max_val <= 65535:
            bit_depth = "16-bit"
        else:
            bit_depth = "32-bit float"

        print(f"  {modality} | Min: {min_val:.2f} | Max: {max_val:.2f} | "
              f"Unique values: {unique_vals} | Bit-depth: {bit_depth} | "
              f"Dtype: {data.dtype}")

Determining bit depth:
* 8-bit → pixel values range 0–255 (256 levels)
* 16-bit → pixel values range 0–65535 (65,536 levels)
* 32-bit float → beyond that, floating point precision

false contouring: if we process a 16-bit image as if it were 8-bit,we get False Contouring — artificial bands/edges appearing in smooth gradient areas of the MRI, like a topographic map effect. Knowing the bit-depth upfront prevents that.


### Step 3: Filtering

firstly we will visualise raw slices for all 25 patients  before filtering so we have a "before" ref

In [ ]:

# Calculate the number of rows needed for all patients and adjust figsize dynamically
num_patients = len(images.keys())
num_modalities = len(modalities_to_process)
fig_height = 5 * num_patients  # Adjust height based on number of patients

fig, axes = plt.subplots(num_patients, num_modalities, figsize=(20, fig_height))

for i, patient in enumerate(sorted(images.keys())):
    for j, modality in enumerate(modalities_to_process):
        # Construct the correct key for accessing the image data
        image_data_key = f"{patient}-{modality}"
        raw_slice = images[patient][image_data_key]  # Pre-sliced dynamically at loading
        raw_normalized = (raw_slice - raw_slice.min()) / (raw_slice.max() - raw_slice.min() + 1e-8)

        axes[i][j].imshow(raw_normalized, cmap='gray')
        axes[i][j].set_title(f'{patient}\n{modality.upper()}')
        axes[i][j].axis('off')

plt.suptitle('Original MRI Slices (Before Processing) — All Patients', fontsize=16)
plt.tight_layout()
plt.show()

applying filters : gaussian, mean and median

for each patient, unkay 4 brain mri scans hain. output will show each patients 4 scans along with their mean, median and gussian images after applyinf those filters.

In [ ]:
# Store filtered results for PSNR/SSIM later
filtered_results = {}

for patient in sorted(images.keys()):
    filtered_results[patient] = {}

    fig, axes = plt.subplots(4, 4, figsize=(20, 20))

    for j, modality in enumerate(modalities_to_process):
        # Construct the correct key for accessing the image data
        image_data_key = f"{patient}-{modality}"
        raw_slice = images[patient][image_data_key]  # Pre-sliced dynamically at loading
        raw_normalized = (raw_slice - raw_slice.min()) / (raw_slice.max() - raw_slice.min() + 1e-8)

        # Apply filters
        gaussian_filtered = gaussian_filter(raw_normalized, sigma=1)
        mean_filtered = uniform_filter(raw_normalized, size=3)
        median_filtered = median_filter(raw_normalized, size=3)

        # Store for later
        filtered_results[patient][modality] = {
            'original': raw_normalized,
            'gaussian': gaussian_filtered,
            'mean': mean_filtered,
            'median': median_filtered
        }

        # Plot
        for k, (name, img) in enumerate([('Original', raw_normalized),
                                          ('Gaussian', gaussian_filtered),
                                          ('Mean', mean_filtered),
                                          ('Median', median_filtered)]):
            axes[j][k].imshow(img, cmap='gray')
            axes[j][k].set_title(f'{modality.upper()} - {name}')
            axes[j][k].axis('off')

    plt.suptitle(f'Before/After Filtering — {patient}', fontsize=14)
    plt.tight_layout()
    plt.show()
    print(f"Done: {patient}")

### Step 4: PSNR and SSIM metrics to quantify how well each filter restored the image

the next cells o/p shows the PSNR and SSIM for each patients 4 MRI slices, and further the PSNR and SSIM values for each of gaussian, mean and median filters.

In [ ]:
#calculate PSNR and SSIM for all patients and modalities

print(f"{'Patient':<25} {'Modality':<10} {'Filter':<12} {'PSNR (dB)':<12} {'SSIM':<10}")
print("=" * 70)

for patient in sorted(filtered_results.keys()):
    for modality in modalities_to_process:
        data = filtered_results[patient][modality]
        original = data['original']
        data_range = original.max() - original.min()

        for filter_name in ['gaussian', 'mean', 'median']:
            filtered = data[filter_name]

            psnr_val = psnr(original, filtered, data_range=data_range)
            ssim_val = ssim(original, filtered, data_range=data_range)

            print(f"{patient:<25} {modality:<10} {filter_name:<12} {psnr_val:<12.4f} {ssim_val:<10.4f}")

        print("-" * 70)

PSNR (Peak Signal-to-Noise Ratio)
Measures how much quality was lost after filtering compared to the original.
* Higher = less distortion = filter preserved the image better
* Below 20 dB = bad, 30-40 dB = good, above 40 dB = excellent

SSIM (Structural Similarity Index)
Measures whether the filtered image still looks structurally similar to the original — edges, textures, patterns.

* Ranges from 0 to 1, closer to 1 = better
* Basically asks "does it still look like the same brain?"

### Step 5: Anti-Aliasing Challenge (applying Gaussian pre-filter before downsampling to avoid the Checkerboard Effect)

pehlay patient 26 ka eik slice with and w/o antialisaing, then unka second brain slice then 3rd then 4th. then 2nd patient and so on

In [ ]:
downsampled_slices = {}  # Global dictionary to store downsampled anti-aliased slices

for patient in sorted(images.keys()):
    downsampled_slices[patient] = {}
    fig, axes = plt.subplots(4, 3, figsize=(15, 20))

    for j, modality in enumerate(modalities_to_process):
        # Construct the correct key for accessing the image data
        image_data_key = f"{patient}-{modality}"
        cleaned_slice = filtered_results[patient][modality]['median']

        # Without anti-aliasing
        downsampled_no_aa = resize(cleaned_slice, (120, 120), anti_aliasing=False, preserve_range=True)

        # With anti-aliasing (Gaussian pre-filter first)
        gaussian_prefiltered = gaussian_filter(cleaned_slice, sigma=1)
        downsampled_with_aa = resize(gaussian_prefiltered, (120, 120), anti_aliasing=True, preserve_range=True)

        # Save the anti-aliased downsampled slice for downstream assignments
        downsampled_slices[patient][modality] = downsampled_with_aa

        axes[j][0].imshow(cleaned_slice, cmap='gray')
        axes[j][0].set_title(f'{modality.upper()} - Cleaned (240x240)')
        axes[j][0].axis('off')

        axes[j][1].imshow(downsampled_no_aa, cmap='gray')
        axes[j][1].set_title(f'{modality.upper()} - No Anti-Aliasing (120x120)')
        axes[j][1].axis('off')

        axes[j][2].imshow(downsampled_with_aa, cmap='gray')
        axes[j][2].set_title(f'{modality.upper()} - With Anti-Aliasing (120x120)')
        axes[j][2].axis('off')

    plt.suptitle(f'Anti-Aliasing Challenge (Cleaned Input) — {patient}', fontsize=14)
    plt.tight_layout()
    plt.show()
    print(f"Done: {patient}")

food for thought:

why is it that the pics w/o anti aliasing look sharper and so called "better" as opposed to pics w/ anti aliasing? i thought pics with anti alisiang are better?

* thing is, that sharpness is FAKE. When we downsample from 240×240 to 120×120, we're throwing away half the pixels. Without a prefilter, the algorithm randomly picks which pixels to keep, this creates artificial sharp edges and patterns that weren't in the original image. That "sharpness" is actually errors and artifacts, not real brain detail.

### Saving before and after pics

In [ ]:
# Create output folders
output_path = '/content/drive/MyDrive/CV_Assignment1_Output_final'
os.makedirs(f'{output_path}/before', exist_ok=True)
os.makedirs(f'{output_path}/after/filtered', exist_ok=True)
os.makedirs(f'{output_path}/after/antialiasing', exist_ok=True)

for patient in sorted(images.keys()):
    for modality in modalities_to_process:
        # Construct the correct key for accessing the image data
        image_data_key = f"{patient}-{modality}"
        raw_slice = images[patient][image_data_key]  # Pre-sliced dynamically at loading
        raw_normalized = (raw_slice - raw_slice.min()) / (raw_slice.max() - raw_slice.min() + 1e-8)

        # --- BEFORE ---
        plt.figure(figsize=(6,6))
        plt.imshow(raw_normalized, cmap='gray')
        plt.title(f'Original - {patient} | {modality.upper()}')
        plt.axis('off')
        plt.savefig(f'{output_path}/before/{patient}_{modality}_original.png', bbox_inches='tight')
        plt.close()

        # --- AFTER: Filters ---
        gaussian_filtered = gaussian_filter(raw_normalized, sigma=1)
        mean_filtered = uniform_filter(raw_normalized, size=3)
        median_filtered = median_filter(raw_normalized, size=3)

        fig, axes = plt.subplots(1, 3, figsize=(18, 6))
        axes[0].imshow(gaussian_filtered, cmap='gray')
        axes[0].set_title('Gaussian Filter')
        axes[0].axis('off')
        axes[1].imshow(mean_filtered, cmap='gray')
        axes[1].set_title('Mean Filter')
        axes[1].axis('off')
        axes[2].imshow(median_filtered, cmap='gray')
        axes[2].set_title('Median Filter')
        axes[2].axis('off')
        plt.suptitle(f'After Filtering - {patient} | {modality.upper()}')
        plt.tight_layout()
        plt.savefig(f'{output_path}/after/filtered/{patient}_{modality}_filtered.png', bbox_inches='tight')
        plt.close()

        # --- AFTER: Anti-Aliasing (Using Median-Filtered Slices as Cleaned Input) ---
        cleaned_slice = filtered_results[patient][modality]['median']
        downsampled_no_aa = resize(cleaned_slice, (120, 120), anti_aliasing=False, preserve_range=True)
        gaussian_prefiltered = gaussian_filter(cleaned_slice, sigma=1)
        downsampled_with_aa = resize(gaussian_prefiltered, (120, 120), anti_aliasing=True, preserve_range=True)

        fig, axes = plt.subplots(1, 3, figsize=(18, 6))
        axes[0].imshow(cleaned_slice, cmap='gray')
        axes[0].set_title('Cleaned (240x240)')
        axes[0].axis('off')
        axes[1].imshow(downsampled_no_aa, cmap='gray')
        axes[1].set_title('Without Anti-Aliasing (120x120)')
        axes[1].axis('off')
        axes[2].imshow(downsampled_with_aa, cmap='gray')
        axes[2].set_title('With Anti-Aliasing (120x120)')
        axes[2].axis('off')
        plt.suptitle(f'Anti-Aliasing - {patient} | {modality.upper()}')
        plt.tight_layout()
        plt.savefig(f'{output_path}/after/antialiasing/{patient}_{modality}_antialiasing.png', bbox_inches='tight')
        plt.close()

        print(f"Saved: {patient} | {modality}")

print("\nAll images saved to Google Drive successfully!")

# ***E-assignment 02***

### Step 1. Segmentation: Edge Detection (Canny & Sobel)

In [ ]:
output_path = '/content/drive/MyDrive/CV_Assignment2_Output/edge_detection'
os.makedirs(output_path, exist_ok=True)
canny_edge_masks = {}

if 'dropped_slices' not in locals() and 'dropped_slices' not in globals():
    dropped_slices = []

for patient_id in sorted(downsampled_slices.keys()):
    # Skip if patient is already dropped entirely
    if any(d['Patient'] == patient_id and d['Modality'] == 'ALL' for d in dropped_slices):
        continue

    patient_path = os.path.join(dataset_path, patient_id)
    canny_edge_masks[patient_id] = {}

    # Load seg file
    seg_files = [f for f in os.listdir(patient_path) if 'seg' in f]
    if not seg_files:
        print(f"Skipping {patient_id}: No segmentation file found.")
        for modality in modalities_to_process:
            dropped_slices.append({
                'Patient': patient_id,
                'Modality': modality.upper(),
                'Slice_Idx': selected_slices[patient_id],
                'Reason': "No segmentation file found."
            })
        continue

    seg_data = nib.load(os.path.join(patient_path, seg_files[0])).get_fdata()
    seg_slice_idx = selected_slices[patient_id]
    seg_slice = seg_data[:, :, seg_slice_idx]
    tumor_region = (seg_slice > 0).astype(float)

    # Resize mask to 120x120 (using order=0 for binary masks)
    tumor_region_resized = resize(tumor_region, (120, 120), order=0, anti_aliasing=False, preserve_range=True)
    tumor_area = np.sum(tumor_region_resized > 0)

    # Check if tumor area is less than 50 pixels (resolution 120x120)
    if tumor_area < 50:
        print(f"⚠️ Dropping patient {patient_id}: Tumor area in selected slice {seg_slice_idx} is too small ({tumor_area} pixels).")
        for modality in modalities_to_process:
            dropped_slices.append({
                'Patient': patient_id,
                'Modality': modality.upper(),
                'Slice_Idx': seg_slice_idx,
                'Reason': f"Tumor area is too small ({tumor_area} pixels) on 120x120 mask."
            })
        continue

    for modality in modalities_to_process:
        raw_normalized = downsampled_slices[patient_id][modality]
        tumor_pixels = raw_normalized * tumor_region_resized

        canny_edges = canny(tumor_pixels, sigma=2)
        sobel_edges = sobel(tumor_pixels)
        sobel_binary = sobel_edges > 0.05

        canny_edge_masks[patient_id][modality] = canny_edges

        fig, axes = plt.subplots(1, 3, figsize=(18, 6))
        axes[0].imshow(tumor_pixels, cmap='gray')
        axes[0].set_title('Tumor Region (120x120)')
        axes[0].axis('off')
        axes[1].imshow(canny_edges, cmap='gray')
        axes[1].set_title('Canny Edge Detection')
        axes[1].axis('off')
        axes[2].imshow(sobel_binary, cmap='gray')
        axes[2].set_title('Sobel Edge Detection')
        axes[2].axis('off')

        plt.suptitle(f'Edge Detection (120x120) — {patient_id} | {modality.upper()} | Slice {seg_slice_idx}', fontsize=14)
        plt.tight_layout()
        plt.savefig(f'{output_path}/{patient_id}_{modality}_edges.png', bbox_inches='tight')
        plt.show()
        plt.close()

    print(f"Done: {patient_id}")

print("\nAll patients saved!")

### Step 2: Morphological Cleaning

This script takes the binary edge masks from the previous step and applies a series of operations to create a solid, clean boundary for the next stage (Chain Coding).

In [ ]:
import os
import numpy as np
import nibabel as nib
import matplotlib.pyplot as plt
from skimage.morphology import disk, erosion, dilation, opening, closing
from skimage.transform import resize
from scipy.ndimage import binary_fill_holes

# Use a disk structuring element of radius 2 to bridge gaps and clean boundary noise
se = disk(2)

cleaned_morph_masks = {}

for patient_id in sorted(downsampled_slices.keys()):
    # Skip if patient is dropped entirely
    if any(d['Patient'] == patient_id and d['Modality'] == 'ALL' for d in dropped_slices):
        continue

    cleaned_morph_masks[patient_id] = {}
    print(f"📊 Morphological Comparison & Cleaning: {patient_id}")

    fig, axes = plt.subplots(len(modalities_to_process), 5, figsize=(18, 3.5 * len(modalities_to_process)))

    for mod_idx, modality in enumerate(modalities_to_process):
        # Skip if this modality is already dropped
        if any(d['Patient'] == patient_id and d['Modality'] == modality.upper() for d in dropped_slices):
            continue

        # 1. Get the Canny edge detection mask from previous step
        canny_mask = canny_edge_masks[patient_id][modality]

        # 2. Fill holes to create a solid binary mask representing the tumor body
        filled_mask = binary_fill_holes(canny_mask)

        # 3. Apply Binary Morphological operations to compare
        eroded  = erosion(filled_mask, se)
        dilated = dilation(filled_mask, se)
        opened  = opening(filled_mask, se)
        closed  = closing(filled_mask, se)

        # 4. Save Closing as the Master Morphological Mask for downstream tasks
        # Rationale: Closing bridges Canny edge boundary gaps and fills small noise holes
        # without shrinking the tumor (like Erosion/Opening) or overestimating it (like Dilation).
        cleaned_morph_masks[patient_id][modality] = closed

        current_axes = axes[mod_idx] if len(modalities_to_process) > 1 else axes

        current_axes[0].imshow(filled_mask, cmap='gray')
        current_axes[0].set_title(f"{modality.upper()} - Filled Mask")
        current_axes[0].axis('off')

        current_axes[1].imshow(eroded, cmap='gray')
        current_axes[1].set_title("Binary Erosion")
        current_axes[1].axis('off')

        current_axes[2].imshow(dilated, cmap='gray')
        current_axes[2].set_title("Binary Dilation")
        current_axes[2].axis('off')

        current_axes[3].imshow(opened, cmap='gray')
        current_axes[3].set_title("Binary Opening")
        current_axes[3].axis('off')

        current_axes[4].imshow(closed, cmap='gray')
        current_axes[4].set_title("Binary Closing (Master)")
        current_axes[4].axis('off')

    plt.suptitle(f"Morphological Operations Comparison ({patient_id})\n*Closing is selected as it bridges boundary gaps and fills noise holes while preserving tumor area.", fontsize=13, y=1.02)
    plt.tight_layout()
    plt.show()

print("\n✅ Morphological processing complete! Master binary masks are stored in 'cleaned_morph_masks'.")

### Step 3 and 4: Boundary representation, and Derive first difference and shape number

The Strategy
1. Selection: We will use the Closing mask from the previous step, as it provides the most continuous boundary.
2. Contour Extraction: We use find_contours to get the $(x, y)$ coordinates of the tumor boundary.
3. Encoding: We convert the movement between consecutive coordinates into a sequence of numbers from 0 to 7.


---



*Q) kis basis par brain par particular areas highlight ho rahay hain? likw whats the reasoning behind it?*

 The boundaries are highlighted based on the mathematical detection of Intensity Gradients and the physical properties of the brain tissues captured by the MRI scanner.

REASONING:

1. Mathematical Basis: Intensity Gradients
At its simplest level, the computer does not know what a "tumor" is; it only sees numbers. The Canny and Sobel filters work by calculating the gradient (the rate of change) between neighboring pixels.

* Edges = Sudden Change: If a pixel has a value of 10 (dark) and the pixel next to it has a value of 200 (bright), the computer calculates a high gradient at that point.

* The Threshold: The "highlighting" occurs where these changes are sharpest. The algorithm marks these locations as "edges."

2. Biological Basis: Tissue Contrast
The reason these gradients happen in the first place is due to how different tissues respond to the MRI's magnetic field and pulses.

* Healthy vs. Pathological: Healthy brain tissue (gray matter, white matter, CSF) has very specific, uniform intensity ranges. Tumors, however, often contain more fluid (edema) or have high concentrations of contrast agents (in T1c scans).

* Modalities: * T1c (Contrast Enhanced): Highlights the "active" tumor core because the contrast agent leaks into the tumor through broken blood vessels.

* T2/FLAIR: Highlights the "edema" or swelling around the tumor, which appears very bright compared to the rest of the brain.

3. Structural Basis: Morphological Continuity
The raw edges found by the filters are often messy or "broken." The Morphological Cleaning step (Dilation and Closing) acts as a bridge.

* Grouping: It assumes that if many edge pixels are close together, they belong to the same structure.

* Gap Filling: By thickening these pixels and then thinning them back down, the computer "heals" gaps in the boundary. The final highlighted red line is the result of the computer tracing the outermost perimeter of these connected groups.


---

To ensure our tumor descriptors are robust against the orientation of the patient's head in the MRI scanner (tilted scans), we need to transform the Absolute Chain Code into a Shape Number.

This process involves two mathematical steps: the First Difference (to achieve rotation invariance) and Normalization (to achieve starting-point invariance).


1. The First Difference (Rotation Invariance): The chain code we generated earlier (e.g., $0, 0, 7, 6...$) depends on the absolute orientation of the object. If the tumor rotates, every digit in the code changes.The First Difference counts the number of 45-degree steps required to get from one direction to the next in a counter-clockwise manner.
* Formula: $d_i = (c_i - c_{i-1}) \pmod 8$

* For the first element ($d_1$), we use the last element of the code as $c_{i-1}$ to treat the boundary as a closed loop.

2. The Shape Number (Starting-Point Invariance):
Even with the First Difference, if we start tracing the tumor from a different pixel, the sequence of numbers will be shifted. The Shape Number is defined as the smallest integer magnitude (lexicographical minimum) of all circular shifts of the First Difference.

* By finding the "smallest" version of the code, we ensure that no matter where the computer starts "looking" at the tumor, the resulting descriptor is identical.

In [ ]:
# 1. CORE CHAIN CODING ALGORITHMS
def get_chain_code(contour):
    directions = {
        (0, 1): 0, (-1, 1): 1, (-1, 0): 2, (-1, -1): 3,
        (0, -1): 4, (1, -1): 5, (1, 0): 6, (1, 1): 7
    }
    chain = []
    for i in range(len(contour) - 1):
        dy = int(round(contour[i+1][0] - contour[i][0]))
        dx = int(round(contour[i+1][1] - contour[i][1]))
        dy = max(-1, min(1, dy))
        dx = max(-1, min(1, dx))
        if (dy, dx) in directions:
            chain.append(directions[(dy, dx)])
    return chain

def get_first_difference(chain):
    return [(chain[i+1] - chain[i]) % 8 for i in range(len(chain)-1)]

def get_shape_number(first_diff):
    doubled = first_diff * 2
    min_val = first_diff.copy()
    for i in range(len(first_diff)):
        rotated = doubled[i:i+len(first_diff)]
        if rotated < min_val:
            min_val = rotated
    return min_val

# 2. OUTPUT CONFIGURATION
output_path = '/content/drive/MyDrive/CV_Assignment2_Output/chaincode'
os.makedirs(output_path, exist_ok=True)

csv_path = os.path.join(output_path, 'chaincode_data.csv')
csv_file = open(csv_path, 'w', newline='')
writer = csv.writer(csv_file)
writer.writerow(['Patient', 'Modality', 'Slice', 'Chain_Code_First20', 'First_Difference_First20', 'Shape_Number_First20'])

# 3. SEQUENTIAL PIPELINE PROCESSING
for patient_id in sorted(downsampled_slices.keys()):
    # Skip if patient is dropped entirely
    if any(d['Patient'] == patient_id and d['Modality'] == 'ALL' for d in dropped_slices):
        continue

    seg_slice_idx = selected_slices[patient_id]
    print(f"📈 Extracting Contour Descriptors for: {patient_id}")

    for modality in modalities_to_process:
        # Skip if this patient-modality is dropped
        if any(d['Patient'] == patient_id and d['Modality'] == modality.upper() for d in dropped_slices):
            continue

        raw_normalized = downsampled_slices[patient_id][modality]
        binary_mask = cleaned_morph_masks[patient_id][modality]

        # Find the boundary perimeter interface
        contours = find_contours(binary_mask, 0.5)
        if len(contours) == 0:
            print(f"⚠️ Dropping {patient_id} | {modality.upper()}: No contours found.")
            dropped_slices.append({
                'Patient': patient_id,
                'Modality': modality.upper(),
                'Slice_Idx': seg_slice_idx,
                'Reason': "No contours found in morphological mask."
            })
            continue

        largest = max(contours, key=len)
        chain = get_chain_code(largest)
        if len(chain) < 2:
            print(f"⚠️ Dropping {patient_id} | {modality.upper()}: Chain code length too short ({len(chain)}).")
            dropped_slices.append({
                'Patient': patient_id,
                'Modality': modality.upper(),
                'Slice_Idx': seg_slice_idx,
                'Reason': f"Chain code too short (length: {len(chain)})."
            })
            continue

        first_diff = get_first_difference(chain)
        shape_num = get_shape_number(first_diff)

        # Truncate to first 20 descriptors
        chain_str = "".join(map(str, chain[:20]))
        diff_str = "".join(map(str, first_diff[:20]))
        shape_str = "".join(map(str, shape_num[:20]))

        # Save record entries to CSV spreadsheet database
        writer.writerow([patient_id, modality, seg_slice_idx, chain_str, diff_str, shape_str])

        # Plot structural overlays
        fig, axes = plt.subplots(1, 2, figsize=(12, 5))

        axes[0].imshow(raw_normalized, cmap='gray')
        axes[0].plot(largest[:, 1], largest[:, 0], 'r-', linewidth=1.5)
        axes[0].set_title('Tumor Contour Overlay (120x120 Canvas)')
        axes[0].axis('off')

        axes[1].imshow(binary_mask, cmap='gray')
        axes[1].set_title('Binary Closing Master Mask')
        axes[1].axis('off')

        plt.suptitle(f'Chain Code Descriptors — {patient_id} | {modality.upper()} | Slice {seg_slice_idx}', fontsize=12, y=1.02)
        plt.tight_layout()

        fig_save_path = os.path.join(output_path, f'{patient_id}_{modality}_chaincode.png')
        plt.savefig(fig_save_path, bbox_inches='tight')
        plt.show()
        plt.close()

        print(f"  ↳ {modality.upper()} | Chain (First 20): {chain_str}")
        print(f"  ↳ {modality.upper()} | First Diff:       {diff_str}")
        print(f"  ↳ {modality.upper()} | Shape Number:     {shape_str}\n")

    print(f"Done Chain Code for: {patient_id}\n")

csv_file.close()
print(f"✅ All patient profiles done! Metrics compiled completely into: '{csv_path}'")

### Step 5: computational geometry

For the Computational Geometry stage, we focus on defining the Convex Hull of the tumor. the goal is to find the smallest convex polygon that completely encloses the detected tumor boundary.

In medical terms, the "Convex Hull" helps quantify the solidity of a tumor. A highly irregular tumor (like a Glioblastoma) will have a much larger Convex Hull area compared to its actual area, whereas a smooth, rounded tumor will fit its hull almost perfectly.





---
The output of the Computational Geometry step consists of 25 consolidated image files (one for each patient) that visually prove the computer has successfully "enclosed" the tumor.


In [ ]:
# 1. SETUP OUTPUT DIRECTORIES
hull_output_path = '/content/drive/MyDrive/CV_Assignment2_Output/convexhull'
os.makedirs(hull_output_path, exist_ok=True)

# Dictionary to hold Convex Hull metrics
convex_hull_data = {}

# 2. PIPELINE ITERATION OVER ALL PATIENTS
for patient_id in sorted(downsampled_slices.keys()):
    # Skip if patient is dropped entirely
    if any(d['Patient'] == patient_id and d['Modality'] == 'ALL' for d in dropped_slices):
        continue

    convex_hull_data[patient_id] = {}
    seg_slice_idx = selected_slices[patient_id]
    print(f"🔷 Computing Convex Hull Overlays for: {patient_id}")

    for modality in modalities_to_process:
        # Skip if this patient-modality is dropped
        if any(d['Patient'] == patient_id and d['Modality'] == modality.upper() for d in dropped_slices):
            continue

        raw_normalized = downsampled_slices[patient_id][modality]
        binary_mask = cleaned_morph_masks[patient_id][modality]

        # Find the contour perimeter
        contours = find_contours(binary_mask, 0.5)
        if len(contours) == 0:
            print(f"⚠️ Dropping {patient_id} | {modality.upper()}: No contours found for Convex Hull.")
            dropped_slices.append({
                'Patient': patient_id,
                'Modality': modality.upper(),
                'Slice_Idx': seg_slice_idx,
                'Reason': "No contours found for Convex Hull."
            })
            continue

        main_contour = max(contours, key=len)

        # 3. COMPUTE THE CONVEX HULL
        try:
            hull = ConvexHull(main_contour)
            hull_pts = main_contour[hull.vertices]
            hull_plot = np.vstack((hull_pts, hull_pts[0]))
        except Exception as e:
            print(f"⚠️ Dropping {patient_id} | {modality.upper()}: Convex Hull calculation failed: {str(e)}.")
            dropped_slices.append({
                'Patient': patient_id,
                'Modality': modality.upper(),
                'Slice_Idx': seg_slice_idx,
                'Reason': f"Convex Hull calculation failed: {str(e)}."
            })
            continue

        # Track hull metrics
        convex_hull_data[patient_id][modality] = {
            'hull_area': hull.volume,       # In 2D, hull.volume represents the area of the polygon
            'hull_perimeter': hull.area,    # In 2D, hull.area represents the perimeter of the polygon
            'vertices': hull_pts
        }

        # 4. VISUALIZATION AND SAVING
        fig, axes = plt.subplots(1, 2, figsize=(12, 5))

        axes[0].imshow(raw_normalized, cmap='gray')
        axes[0].plot(main_contour[:, 1], main_contour[:, 0], 'r-', linewidth=1.2, label='Tumor Border')
        axes[0].plot(hull_plot[:, 1], hull_plot[:, 0], 'cyan', linewidth=2, label='Convex Hull')
        axes[0].set_title('Convex Hull Minimal Enclosure')
        axes[0].axis('off')
        axes[0].legend(loc='upper right')

        axes[1].imshow(binary_mask, cmap='gray')
        axes[1].set_title('Binary Closing Source Mask')
        axes[1].axis('off')

        plt.suptitle(f'Deliverable: Convex Hull Geometry — {patient_id} | {modality.upper()} | Slice {seg_slice_idx}', fontsize=12, y=1.02)
        plt.tight_layout()

        fig_save_path = os.path.join(hull_output_path, f'{patient_id}_{modality}_convexhull.png')
        plt.savefig(fig_save_path, bbox_inches='tight')
        plt.show()
        plt.close()

        print(f"  ↳ {modality.upper()} | Tumor Points: {len(main_contour)} | Hull Vertices: {len(hull_pts)}")

    print(f"Done Convex Hull for: {patient_id}\n")

print(f"✅ Convex Hull wrapping sequence completed! Images logged at: '{hull_output_path}'")

In [ ]:
# Display a summary table of dropped slices/modalities
from IPython.display import HTML, display
import pandas as pd

if len(dropped_slices) == 0:
    print("🎉 All slices and modalities passed quality checks. No slices were dropped!")
else:
    print(f"⚠️ A total of {len(dropped_slices)} patient-modality slices were dropped. Detailed log:")
    df_dropped = pd.DataFrame(dropped_slices)
    display(HTML(df_dropped.to_html(index=False)))


# Deleting garbage data to free up RAM

In [ ]:
import gc
# This deletes the giant dictionary holding all 3D volumes
if 'images' in locals():
    del images
# Manually trigger garbage collection to empty the RAM
gc.collect()

# ***E-assignment 03***

### Step 1, 2 and 3: GLCM matrices, statistical descriptors and geometrical features

*Before we compute the matrices, its important to know that before this, i did assignment 3 by running the full assignment 2 and 1 pipeline, which caused my colab runtime to shoot upto approx 10 gb RAM usage out of 12, so i did assignment 3 again, but this time, i deleted garbage stuff that was saved, and in the following GLCM code, i run a loop which loads and scans data from drive, normalizes it, finds boondary and convex hull, computes glcm and saves the numbers into a list and immediatelty deletes heavy images from RAM. after this, a final_features.csv file will be saved to my drive, so for next sessions, i dont have to load all 3d volumes and run any canny or any such filters, i only need to run csv*


---



The Gray-Level Co-occurrence Matrix (GLCM) tracks how often pairs of pixels with specific values occur in a specific spatial relationship

how we'll proceed to calc GLCM matrices:
1. To compute the GLCM only for the tumor, we must first "fill" the boundary we found in Assignment 2 to create a solid binary mask. We then apply this mask to the original MRI slice.
2. Because MRI data often has high bit-depth (12-bit or 16-bit), we must first quantize the tumor pixels to a smaller number of gray levels (usually 8 or 16). If we use 256 levels, the matrix becomes too sparse and loses its statistical power.

Statistical descriptors mentioned in next step (energy, entropy, contrast) will also be calculated here, geometrical features (area, perim etc) will also be calculated in same code block and saved in csv



In [ ]:
import gc
my_path = dataset_path

patient_list = sorted([f for f in os.listdir(my_path) if f.startswith('BraTS')])
final_features = []
modalities_to_process = ['t1n', 't1c', 't2w', 't2f']

print(f"🚀 Extracting Texture and Geometric Features for active patients...")

for p_id in patient_list:
    if p_id not in downsampled_slices:
        continue
    # Skip if patient is dropped entirely
    if any(d['Patient'] == p_id and d['Modality'] == 'ALL' for d in dropped_slices):
        continue

    seg_slice_idx = selected_slices[p_id]

    for mod in modalities_to_process:
        # Skip if this patient-modality is dropped
        if any(d['Patient'] == p_id and d['Modality'] == mod.upper() for d in dropped_slices):
            continue

        # Use the 120x120 downsampled slice and 120x120 Closed morphological binary mask
        norm = downsampled_slices[p_id][mod]
        mask_for_features = cleaned_morph_masks[p_id][mod].astype(int)

        # 3. GEOMETRIC FEATURES (Task 3)
        labeled_mask = label(mask_for_features)
        props = regionprops(labeled_mask)

        if props:
            p = max(props, key=lambda prop: prop.area)
            # Double check area threshold on morphological mask
            if p.area < 50:
                print(f"⚠️ Dropping {p_id} | {mod.upper()}: Tumor area {p.area} is too small on labeled mask.")
                dropped_slices.append({
                    'Patient': p_id,
                    'Modality': mod.upper(),
                    'Slice_Idx': seg_slice_idx,
                    'Reason': f"Tumor area in morphological mask is too small ({p.area} pixels)."
                })
                continue

            area = p.area
            centroid = p.centroid # Returns (y, x)
            perimeter = p.perimeter
            circularity = (4 * np.pi * area) / (perimeter**2) if perimeter > 0 else 0

            # 4. TEXTURE FEATURES (Tasks 1 & 2)
            # Quantize only the relevant tumor intensity values for GLCM
            tumor_region_intensity = norm * mask_for_features
            quantized = (tumor_region_intensity * 15).astype(np.uint8)

            # Ensure non-zero pixels for GLCM calculation
            glcm_input_region = np.where(mask_for_features, quantized, 0)

            if np.sum(glcm_input_region > 0) < 2:
                 print(f"⚠️ Dropping {p_id} | {mod.upper()}: Not enough pixels for GLCM calculation.")
                 dropped_slices.append({
                     'Patient': p_id,
                     'Modality': mod.upper(),
                     'Slice_Idx': seg_slice_idx,
                     'Reason': "Not enough pixels for GLCM calculation."
                 })
                 continue

            # GLCM levels should match the range of quantized values (0-15 -> 16 levels)
            glcm = graycomatrix(glcm_input_region, distances=[1], angles=[0], levels=16, symmetric=True, normed=True)

            energy = graycoprops(glcm, 'energy')[0, 0]
            contrast = graycoprops(glcm, 'contrast')[0, 0]
            entropy = -np.sum(glcm * np.log2(glcm + 1e-10)) # Add a small epsilon to avoid log(0)

            # 5. Store everything together
            final_features.append({
                'Patient': p_id,
                'Modality': mod.upper(),
                'Slice_Idx': seg_slice_idx,
                'Area': area,
                'Centroid_Y': centroid[0],
                'Centroid_X': centroid[1],
                'Perimeter': perimeter,
                'Circularity': circularity,
                'Energy': energy,
                'Contrast': contrast,
                'Entropy': entropy
            })
        else:
            print(f"⚠️ Dropping {p_id} | {mod.upper()}: No tumor region found in morphological mask.")
            dropped_slices.append({
                'Patient': p_id,
                'Modality': mod.upper(),
                'Slice_Idx': seg_slice_idx,
                'Reason': "No tumor region found after labeling morphological mask."
            })

    gc.collect()

# Save one final CSV with ALL features
df_final = pd.DataFrame(final_features)
save_path = os.path.join(my_path, 'BraTS_Complete_Feature_Vector.csv')
df_final.to_csv(save_path, index=False)

print(f"✅ FINAL SUCCESS: All features saved to {save_path}")
display(df_final.head())

### Step 4: Traditional Classification

In Traditional Classification, we stop looking at pixels and start looking at numbers to make a diagnosis. Here is exactly what this step involves:

The classification step is where we teach the computer that:

* Malignant tumors usually have high entropy (chaotic texture) and low circularity (irregular shape).

* Benign tumors usually have low entropy (smooth texture) and high circularity (round shape).



---
what we need to do in this step:
1. For a computer to learn, it needs to know the "right answer" for the training data.we need a column in your CSV called Label.
Since we are using the BraTS dataset, the labels are usually provided in the metadata or based on the patient IDs (e.g., High-Grade Glioma vs. Low-Grade Glioma) but since is dataset mein labels arent provided, we will use unsupervised learning.
2. we will use K means clustering, for classifiying the tumors as malignant/benign, we will use the following philosophy:

* Cluster with High Entropy & Low Circularity: We label this as "Malignant" (Complex/Irregular).

* Cluster with Low Entropy & High Circularity: We label this as "Benign" (Smooth/Uniform).


In [ ]:
# 1. Load your complete feature vector
df = pd.read_csv('/content/drive/MyDrive/BraTS_Complete_Feature_Vector.csv')

# 2. Select the features for clustering
features = ['Area', 'Perimeter', 'Circularity', 'Energy', 'Contrast', 'Entropy']
X = df[features]

# 3. Scaling (CRITICAL: K-Means needs all numbers on the same scale)
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

# 4. Run K-Means to find 2 groups
kmeans = KMeans(n_clusters=2, random_state=42, n_init=10)
df['Cluster'] = kmeans.fit_predict(X_scaled)

# 5. Logic to assign "Malignant" vs "Benign"
# We assume the cluster with higher average Entropy is 'Malignant'
cluster_0_entropy = df[df['Cluster'] == 0]['Entropy'].mean()
cluster_1_entropy = df[df['Cluster'] == 1]['Entropy'].mean()

if cluster_1_entropy > cluster_0_entropy:
    df['Label'] = df['Cluster'].map({1: 'Malignant', 0: 'Benign'})
else:
    df['Label'] = df['Cluster'].map({0: 'Malignant', 1: 'Benign'})

# 6. Save the newly labeled dataset
df.to_csv('/content/drive/MyDrive/BraTS_Labeled_Features.csv', index=False)

print("✅ Labels assigned based on feature similarity!")
display(df[['Patient', 'Modality', 'Entropy', 'Circularity', 'Label']].head(10))

In [ ]:
# 1. Load the labeled features generated by K-Means
df_labeled = pd.read_csv('/content/drive/MyDrive/BraTS_Labeled_Features.csv')

# 2. Select features (X) and target labels (y)
features = ['Area', 'Perimeter', 'Circularity', 'Energy', 'Contrast', 'Entropy']
X = df_labeled[features]

# Encode labels: Benign -> 0, Malignant -> 1
le = LabelEncoder()
y = le.fit_transform(df_labeled['Label'])

# 3. Scale the features (CRITICAL: KNN is distance-based, so features must be on the same scale!)
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

# 4. Initialize KNN Classifier (using k=3 neighbors as a robust default)
knn_clf = KNeighborsClassifier(n_neighbors=3)

# 5. Perform 5-Fold Stratified Cross-Validation
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
cv_results = cross_validate(knn_clf, X_scaled, y, cv=cv, scoring=['accuracy', 'precision', 'recall', 'f1'])

# Display Cross-Validation performance metrics
print("=== 5-Fold KNN Cross-Validation Performance ===")
print(f"Mean Accuracy:  {np.mean(cv_results['test_accuracy']):.4f} (± {np.std(cv_results['test_accuracy']):.4f})")
print(f"Mean Precision: {np.mean(cv_results['test_precision']):.4f} (± {np.std(cv_results['test_precision']):.4f})")
print(f"Mean Recall:    {np.mean(cv_results['test_recall']):.4f} (± {np.std(cv_results['test_recall']):.4f})")
print(f"Mean F1-Score:  {np.mean(cv_results['test_f1']):.4f} (± {np.std(cv_results['test_f1']):.4f})")
print("===============================================")

# ***E-final project***

final pipeline:

1.  Acquisition & Preprocessing: Load BraTS-PEDs dataset, bit-depth check, Gaussian/Mean/Median filtering, anti-aliasing
2. Segmentation: Canny edge detection, morphological cleaning
3. Specular Mitigation (bonus research challenge): Detect and remove bright specular spots that interfere with tumor detection
4. YOLO Preparation: Convert segmentation masks to YOLO bounding box format. Prepare train/test split
5. train yolo on prepared dataset
6. evaluation: prec, recall etc



---

since in previous assignments, we have alr done the first 2 steps, well start with specular mitigation

### Applying YOLO for tumor localisation

In [ ]:
yolo_path = '/content/drive/MyDrive/yolo_dataset'
os.makedirs(f'{yolo_path}/images/train', exist_ok=True)
os.makedirs(f'{yolo_path}/images/val', exist_ok=True)
os.makedirs(f'{yolo_path}/labels/train', exist_ok=True)
os.makedirs(f'{yolo_path}/labels/val', exist_ok=True)

def get_yolo_bbox(binary_mask):
    """Convert binary mask to YOLO format bounding box"""
    props = regionprops(label(binary_mask))
    if len(props) == 0:
        return None
    # Get largest region
    largest = max(props, key=lambda r: r.area)
    minr, minc, maxr, maxc = largest.bbox
    h, w = binary_mask.shape

    # YOLO format: class x_center y_center width height (all normalized 0-1)
    x_center = ((minc + maxc) / 2) / w
    y_center = ((minr + maxr) / 2) / h
    width = (maxc - minc) / w
    height = (maxr - minr) / h

    return x_center, y_center, width, height

img_count = 0
patients = sorted(downsampled_slices.keys())
# Filter out patients who are completely dropped
active_patients = [p for p in patients if not all(any(d['Patient'] == p and d['Modality'] == mod.upper() for d in dropped_slices) for mod in modalities_to_process)]
train_patients = active_patients[:20]
val_patients = active_patients[20:]

for patient_id in active_patients:
    split = 'train' if patient_id in train_patients else 'val'
    seg_slice_idx = selected_slices[patient_id]

    for modality in modalities_to_process:
        # Skip if this patient-modality is dropped
        if any(d['Patient'] == patient_id and d['Modality'] == modality.upper() for d in dropped_slices):
            continue

        # DIRECTLY load the downsampled anti-aliased slice (NO specular mitigation!)
        raw_slice = downsampled_slices[patient_id][modality]

        # Get bounding box from Closed Morphological Mask (120x120)
        binary_mask = cleaned_morph_masks[patient_id][modality]
        bbox = get_yolo_bbox(binary_mask)
        if bbox is None:
            continue

        # Save image (120x120)
        img_uint8 = (raw_slice * 255).astype(np.uint8)
        img_name = f'{patient_id}_{modality}'
        cv2.imwrite(f'{yolo_path}/images/{split}/{img_name}.png', img_uint8)

        # Save label
        with open(f'{yolo_path}/labels/{split}/{img_name}.txt', 'w') as f:
            f.write(f'0 {bbox[0]:.6f} {bbox[1]:.6f} {bbox[2]:.6f} {bbox[3]:.6f}\n')

        img_count += 1

print(f"Total images prepared: {img_count}")
print(f"Train active patients: {len(train_patients)}")
print(f"Val active patients: {len(val_patients)}")

### yolo config and training

In [ ]:
model = YOLO('yolov8s.pt')

# Train
results = model.train(
    data=f'{yolo_path}/dataset.yaml',
    epochs=50,
    imgsz=128,              # Match the 120x120 downsampled slice resolution (closest multiple of 32)
    batch=8,                # Train in batches of 8 slices
    name='brain_tumor_yolo',
    project='/content/drive/MyDrive/yolo_results',
    device='cpu',               # Use CPU for training as no GPU is available
    lr0=0.01,               # Initial learning rate
    lrf=0.01,               # Final learning rate fraction (cosine decay factor)
    momentum=0.937,         # Optimizer momentum
    weight_decay=0.0005,    # Regularization weight decay to prevent overfitting
    box=7.5,                # Weight factor for the bounding box loss (CIoU)
    cls=0.5                 # Weight factor for the classification loss (Binary Cross-Entropy)
)

print("Training done!")

### Conf matrix, acc/loss curves

In [ ]:
# Define your dataset and training run output paths
yolo_path = '/content/drive/MyDrive/yolo_dataset'  # Add this line to prevent NameErrors!
run_path = '/content/drive/MyDrive/yolo_results/brain_tumor_yolo-3'

# Define your training run output path
run_path = '/content/drive/MyDrive/yolo_results/brain_tumor_yolo-3' # Corrected path to actual training output

# ==========================================
# 1. DISPLAY LOSS AND ACCURACY (RESULTS.PNG)
# ==========================================
print("=== 1. YOLOv8 Training Loss and Accuracy Curves ===")
results_plot = os.path.join(run_path, 'results.png')
if os.path.exists(results_plot):
    display(Image(filename=results_plot, width=800))
else:
    print(f"Results plot not found at {results_plot}")
print("\n" + "="*50 + "\n")

# ==========================================
# 2. DISPLAY CONFUSION MATRIX
# ==========================================
print("=== 2. YOLOv8 Confusion Matrix ===")
conf_matrix = os.path.join(run_path, 'confusion_matrix_normalized.png')
if not os.path.exists(conf_matrix):
    conf_matrix = os.path.join(run_path, 'confusion_matrix.png')

if os.path.exists(conf_matrix):
    display(Image(filename=conf_matrix, width=600))
else:
    print(f"Confusion matrix image not found at {conf_matrix}")
print("\n" + "="*50 + "\n")

# ==========================================
# 3. DISPLAY A YOLO CLASSIFIED EXAMPLE
# ==========================================
print("=== 3. YOLO Classified Validation Example ===")
val_img_dir = f'{yolo_path}/images/val'
train_img_dir = f'{yolo_path}/images/train'

sample_img_path = None

# Prioritize finding a sample image from the training directory if validation is empty
if os.path.exists(val_img_dir) and len(os.listdir(val_img_dir)) > 0:
    sample_img_name = sorted(os.listdir(val_img_dir))[0]
    sample_img_path = os.path.join(val_img_dir, sample_img_name)
    print("Using a sample from the validation directory for visualization.")
elif os.path.exists(train_img_dir) and len(os.listdir(train_img_dir)) > 0:
    sample_img_name = sorted(os.listdir(train_img_dir))[0]
    sample_img_path = os.path.join(train_img_dir, sample_img_name)
    print("Validation directory is empty. Using a sample from the training directory instead.")
else:
    print("Neither validation nor training directory found or they are empty. Make sure you prepared the dataset first!")


if sample_img_path:
    # Explicitly load the trained model weights for prediction
    trained_model_path = os.path.join(run_path, 'weights', 'best.pt')
    model = YOLO(trained_model_path) # Load the actual trained weights

    # Run prediction using your trained model
    pred_results = model.predict(sample_img_path, device='cpu', verbose=False)

    # Plot predicted box using YOLO's built-in plotting tool
    yolo_rendered = pred_results[0].plot()
    yolo_rendered_rgb = cv2.cvtColor(yolo_rendered, cv2.COLOR_BGR2RGB)

    # Draw side-by-side: Original image vs. YOLO prediction overlay
    fig, axes = plt.subplots(1, 2, figsize=(12, 6))

    # Left: Original Image
    original_img = cv2.imread(sample_img_path)
    original_img_rgb = cv2.cvtColor(original_img, cv2.COLOR_BGR2RGB)
    axes[0].imshow(original_img_rgb)
    axes[0].set_title("Original Sample Image (120x120)")
    axes[0].axis('off')

    # Right: YOLO Detection Bounding Box Overlay
    axes[1].imshow(yolo_rendered_rgb)
    axes[1].set_title("YOLO Detection (Bounding Box + Confidence)")
    axes[1].axis('off')

    plt.suptitle(f"Example Classification Model Test: {sample_img_name}", fontsize=14)
    plt.tight_layout()
    plt.show()


**confusion matrix explained:**

***1) Top-Left (0.95 / 95%) — True Positives (TP):***

What it means: Out of all the validation slices that actually contain a tumor, your YOLO model successfully detected and drew a bounding box around 95% of them.
Analysis: This is a very strong score, showing that your model is highly sensitive and rarely misses a tumor.

***2) Bottom-Left (0.05 / 5%) — False Negatives (FN):***

What it means: In 5% of the validation slices where a tumor was present, the model failed to detect it (meaning it predicted it as background).
Analysis: This represents your model's miss rate. It missed only 5% of the tumors in your dataset.

***3) Top-Right (1.00 / 100%) — False Positives (FP):***

When the model looks at the empty background, it has two choices:
* it can make a mistake (draw a false box) or do the right thing (draw nothing).
The confusion matrix only counts the mistakes the model makes on the background. It completely ignores all the times the model correctly did nothing.
Because the chart only keeps track of the mistakes and ignores the correct decisions, 100% of the background data recorded on the chart are mistakes.
That is why it says 1.00 (100%). It is simply showing that 100% of the background events it chose to display are mistakes.
* It does not mean your model is bad or made many errors. Even if the model made only one single false box in the background, that one mistake represents 100% of the background data shown on the chart.

***4) Bottom-Right (Blank / Empty) — True Negatives (TN):***

What it means: This represents regions of background where the model correctly did not predict a tumor.
Why is it blank? In object detection, we do not calculate True Negatives because there is an infinite amount of background space where a model could have drawn a box but correctly chose not to. Because this value is mathematically undefined (NaN), it is left completely blank.

# Extra

In [ ]:
!jupyter nbconvert --to html "/content/drive/MyDrive/Colab Notebooks/CV_Eproject_Fixed_v2_final.ipynb"
